In [ ]:
# reading a dataset that has Feedback in English, Recorded Language
df = spark.read.format("csv").option("header","true").load("Files/products-files/product_feedback_multilingual.csv")

display(df)

In [ ]:
# importing required packages
# note that ml.congnitive is deprecated, recommend to use ml.services

import synapse.ml.core
#from synapse.ml.cognitive.language import AnalyzeText
from synapse.ml.services.language import AnalyzeText
from synapse.ml.services.translate import *
from pyspark.sql.functions import col, flatten


In [ ]:
# configuring the model for language detection

model = (AnalyzeText()
        .setTextCol("Feedback")
        .setKind("LanguageDetection")
        .setOutputCol("response"))

result = model.transform(df)\
        .withColumn("documents", col("response.documents"))\
        .withColumn("detectedLanguage", col("documents.detectedLanguage.name"))

# configuring the model for sentiment analysis
model = (AnalyzeText()
        .setTextCol("Feedback")
        .setKind("SentimentAnalysis")
        .setOutputCol("response"))

result = model.transform(result)\
        .withColumn("documents", col("response.documents"))\
        .withColumn("sentiment", col("documents.sentiment"))

# configuring the model for translation
translate = (Translate()
    .setTextCol("Feedback")
    .setToLanguage(["en"]) # can pass multiple values
    .setOutputCol("response")
    .setConcurrency(5))

result = translate.transform(result)\
        .withColumn("translations", flatten(col("response.translations")))\
        .withColumn("translation", col("translations")[0]["text"]) # since we get only one language


for col_name in result.columns:
    result = result.withColumnRenamed(col_name, col_name.replace(" ", ""))

# saving the result with model outputs
result = result.select("ProductID", "ShopRating", "ServiceRating", "Feedback", "Language", "detectedLanguage", "sentiment", "translation")
result.write.format("delta").mode("overwrite").saveAsTable("ProductFeedback")